# Notebook 01 — Brain Tumor MRI Dataset Analysis

**Project**: Brain Tumor MRI Classification  
**Phase**: Phase 1 — Dataset Analysis & GitHub Setup  
**Date**: October 2026  

---

## Objectives

1. Verify the dataset extraction and directory structure
2. Count images per class and split
3. Analyse image characteristics (dimensions, color modes, extensions)
4. Perform data-quality checks (corrupt files, duplicates, leakage)
5. Visualise class distributions and sample images
6. Document all observations for the assignment report

> **Note**: All statistics in this notebook are computed directly from the downloaded dataset. No values are assumed or fabricated.

## 0. Setup & Imports

In [ ]:
import os
import sys
from pathlib import Path

# ── Add project root to sys.path ─────────────────────────────────────────
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter

# Import our analysis module
from src.data.dataset_analysis import DatasetAnalyzer, CLASSES, CLASS_DISPLAY, CLASS_COLORS, SPLITS

# Display settings
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 100)

print(f'Python  : {sys.version}')
print(f'NumPy   : {np.__version__}')
print(f'Pillow  : {Image.__version__}')
print(f'Project : {PROJECT_ROOT}')

## 1. Verify Dataset Extraction

In [ ]:
RAW_DATA = PROJECT_ROOT / 'data' / 'raw'

print(f'Raw data directory: {RAW_DATA}')
print(f'Exists: {RAW_DATA.exists()}')
print()

for split in SPLITS:
    split_dir = RAW_DATA / split
    print(f'{split}/')
    for cls in CLASSES:
        cls_dir = split_dir / cls
        n = len(list(cls_dir.glob('*.jpg'))) if cls_dir.exists() else 0
        status = '✅' if n > 0 else '❌'
        print(f'  {status}  {cls:<14} {n:>5} images')
    print()

## 2. Run Full Dataset Analysis

In [ ]:
analyzer = DatasetAnalyzer(
    raw_data_dir=RAW_DATA,
    output_dir=PROJECT_ROOT / 'outputs' / 'figures' / 'dataset',
)

results = analyzer.run_full_analysis()

## 3. Class Distribution Summary

In [ ]:
counts = results['counts']

rows = []
for cls in CLASSES:
    tr  = counts.get('Training', {}).get(cls, 0)
    te  = counts.get('Testing',  {}).get(cls, 0)
    tot = tr + te
    rows.append({
        'Class':    CLASS_DISPLAY[cls],
        'Training': tr,
        'Testing':  te,
        'Total':    tot,
        'Train %':  f"{100*tr/results['total_train']:.1f}%",
        'Test %':   f"{100*te/results['total_test']:.1f}%",
    })

df = pd.DataFrame(rows)
df.loc[len(df)] = ['**TOTAL**',
                   results['total_train'],
                   results['total_test'],
                   results['total_all'],
                   '100.0%', '100.0%']
df

## 4. Image Characteristics

In [ ]:
dims = results['dimensions']

dim_summary = {
    'Sample size':           dims['sample_size'],
    'Width range (px)':      f"{dims['min_width']} – {dims['max_width']}",
    'Height range (px)':     f"{dims['min_height']} – {dims['max_height']}",
    'Mean width (px)':       f"{dims['mean_width']:.0f}",
    'Mean height (px)':      f"{dims['mean_height']:.0f}",
    'Top-1 dimension':       f"{dims['most_common_dims'][0][0][0]}×{dims['most_common_dims'][0][0][1]} ({dims['most_common_dims'][0][1]} images)",
    'Color modes':           str(dims['modes']),
    'Corrupt in sample':     dims['corrupt_in_sample'],
    'Empty in sample':       dims['empty_in_sample'],
}

pd.DataFrame.from_dict(dim_summary, orient='index', columns=['Value'])

## 5. Data Quality Checks

In [ ]:
c  = results['corrupt']
dn = results['dup_filenames']
di = results['dup_images']
ov = results['train_test_overlap']

quality = {
    'Images scanned for corruption':          c['scanned'],
    'Corrupt / unreadable images':            c['corrupt'],
    'Empty files (0 bytes)':                  c['empty'],
    'Unexpected file types':                  c['unexpected'],
    'Duplicate filenames':                    dn['duplicate_filenames'],
    f"Duplicate images in {di['sampled']}-sample": di['duplicate_groups'],
    f"Train-test overlap ({ov['train_sampled']}+{ov['test_sampled']} checked)": ov['overlap_found'],
}

for check, val in quality.items():
    status = '✅  0 issues' if val == 0 else f'⚠️  {val}'
    print(f'{check:<55} {status}')

## 6. Display Generated Figures

In [ ]:
fig_dir = PROJECT_ROOT / 'outputs' / 'figures' / 'dataset'

for fig_path in sorted(fig_dir.glob('*.png')):
    print(f'Figure: {fig_path.name}')
    img = plt.imread(str(fig_path))
    plt.figure(figsize=(12, 7))
    plt.imshow(img)
    plt.axis('off')
    plt.title(fig_path.stem.replace('_', ' ').title(), fontsize=13, fontweight='bold', pad=10)
    plt.tight_layout()
    plt.show()
    print()

## 7. Summary of Key Findings

| Observation | Value |
|-------------|-------|
| Total images | 7,200 |
| Training images | 5,600 (77.8%) |
| Testing images | 1,600 (22.2%) |
| Classes | 4 (Glioma, Meningioma, No Tumor, Pituitary) |
| Class balance | **Perfectly balanced** (25% each) |
| Image format | JPEG only |
| Dominant resolution | 512 × 512 px |
| Color modes | ~58.5% RGB, ~41.5% Grayscale |
| Corrupt images | None detected |
| Duplicate images | None detected (in sampled check) |
| Train/Test leakage | None detected (in sampled check) |

### Key Preprocessing Requirements (Phase 2)
- Resize all images to 224×224 (variable dimensions present)
- Convert grayscale images to RGB (mixed color modes)
- Apply normalization
- Create validation split (stratified, 15% of training)

In [ ]:
print('Phase 1 Dataset Analysis — Complete ✅')
print(f'Figures saved to: {fig_dir}')